In [1]:
import json
from pathlib import Path

ROOT = Path(r"D:\projects\f1-project")

for nb_path in sorted((ROOT / "notebooks").glob("*.ipynb")):
    notebook = json.loads(nb_path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        looks_like_fetching = ("def " in source) and (
            "requests" in source or "cache" in source.lower() or "jolpica" in source.lower()
        )
        if looks_like_fetching:
            print("=" * 25, nb_path.name)
            print(source)
            print()

========================= 01_race_data_exploration.ipynb
import requests
import pandas as pd
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1"

def fetch_race_laps(season, round_num, limit=100):
    url = f"{BASE_URL}/{season}/{round_num}/laps/"
    offset = 0
    total = None
    rows = []

    while total is None or offset < total:
        params = {"limit": limit, "offset": offset}
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        mr_data = response.json()["MRData"]
        total = int(mr_data["total"])
        returned_limit = int(mr_data["limit"])
        returned_offset = int(mr_data["offset"])

        races = mr_data["RaceTable"].get("Races", [])
        if not races:
            break

        lap_groups = races[0].get("Laps", [])

        for lap_group in lap_groups:
            lap_number = int(lap_group["number"])

            for timing in lap_group.get("Timings", []):
                rows.append({
         

In [2]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

ROOT = Path(r"D:\projects\f1-project")
RAW_DIR = ROOT / "data" / "raw" / "jolpica"
LAP_CACHE_DIR = RAW_DIR / "laps"
PIT_CACHE_DIR = RAW_DIR / "pit_stops"
PROCESSED_DIR = ROOT / "data" / "processed"
for folder in (LAP_CACHE_DIR, PIT_CACHE_DIR, PROCESSED_DIR):
    folder.mkdir(parents=True, exist_ok=True)

API = "https://api.jolpi.ca/ergast/f1"
SLEEP_SECONDS = 1.0          # pause after every real download
SEASONS = [2023, 2024]
KEYS = ["season", "round", "lap", "driver_id"]


def to_seconds(text):
    """'1:39.019' -> 99.019, '23.4' -> 23.4, None/NaN -> NaN."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    total = 0.0
    for part in str(text).split(":"):
        total = total * 60 + float(part)
    return total


def get_json(url, params, cache_file, max_retries=6):
    """Return the JSON for one API page. Uses the cache if present; otherwise downloads,
    retrying with growing waits if the API says 'too many requests' (429)."""
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
        except requests.RequestException as err:
            print(f"   network error ({err}); retrying")
            time.sleep(5 * 2 ** attempt)
            continue

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After", "")
            wait = float(retry_after) if retry_after.isdigit() else 5 * 2 ** attempt
            print(f"   429 rate limit; waiting {wait:.0f}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        data = response.json()
        cache_file.write_text(json.dumps(data), encoding="utf-8")
        time.sleep(SLEEP_SECONDS)
        return data

    raise RuntimeError(f"Still blocked after {max_retries} tries: {url}")


def get_schedule(season):
    data = get_json(f"{API}/{season}.json", {"limit": 100}, RAW_DIR / f"{season}_schedule.json")
    races = data["MRData"]["RaceTable"]["Races"]
    return pd.DataFrame([
        {"season": season, "round": int(r["round"]), "race_name": r["raceName"], "date": r["date"]}
        for r in races
    ])


def fetch_laps(season, rnd):
    """All lap timings for one race. Moves the offset by the page limit the API reports."""
    url = f"{API}/{season}/{rnd}/laps/"
    rows, offset, total = [], 0, 0
    while True:
        cache_file = LAP_CACHE_DIR / f"{season}_round_{rnd}_laps_offset_{offset}.json"
        data = get_json(url, {"limit": 100, "offset": offset}, cache_file)
        mr = data["MRData"]
        total = int(mr["total"])
        races = mr["RaceTable"].get("Races", [])
        if not races:
            break
        for lap_group in races[0].get("Laps", []):
            lap_number = int(lap_group["number"])
            for timing in lap_group.get("Timings", []):
                rows.append({
                    "season": season, "round": rnd, "lap": lap_number,
                    "driver_id": timing["driverId"],
                    "position": int(timing["position"]),
                    "lap_time": timing["time"],
                })
        offset = int(mr["offset"]) + int(mr["limit"])   # NOT len(lap groups)
        if offset >= total:
            break
    columns = ["season", "round", "lap", "driver_id", "position", "lap_time"]
    return pd.DataFrame(rows, columns=columns), total


def fetch_pit_stops(season, rnd):
    """All pit stops for one race. Keeps the RAW duration text; parsed later with to_seconds."""
    url = f"{API}/{season}/{rnd}/pitstops/"
    rows, offset = [], 0
    while True:
        name = (f"{season}_round_{rnd}_pit_stops.json" if offset == 0
                else f"{season}_round_{rnd}_pit_stops_offset_{offset}.json")
        data = get_json(url, {"limit": 100, "offset": offset}, PIT_CACHE_DIR / name)
        mr = data["MRData"]
        total = int(mr["total"])
        races = mr["RaceTable"].get("Races", [])
        for stop in (races[0].get("PitStops", []) if races else []):
            rows.append({
                "season": season, "round": rnd, "driver_id": stop["driverId"],
                "lap": int(stop["lap"]), "stop": int(stop["stop"]),
                "time": stop.get("time"), "duration_raw": stop.get("duration"),
            })
        offset = int(mr["offset"]) + int(mr["limit"])
        if offset >= total:
            break
    columns = ["season", "round", "driver_id", "lap", "stop", "time", "duration_raw"]
    return pd.DataFrame(rows, columns=columns)


# ---------- main loop ----------
schedule = pd.concat([get_schedule(s) for s in SEASONS], ignore_index=True)
print("Scheduled races:", len(schedule))

existing_laps = pd.read_csv(PROCESSED_DIR / "laps_with_pit_flags_2023_2024_first5.csv")
existing_laps = existing_laps[["season", "round", "lap", "driver_id", "position", "lap_time"]]
already_have = set(zip(existing_laps["season"], existing_laps["round"]))

lap_frames, pit_frames, qc_rows = [existing_laps], [], []
incomplete = False
t0 = time.time()

for race in schedule.itertuples():
    season, rnd = int(race.season), int(race.round)
    try:
        if (season, rnd) not in already_have:
            laps_df, api_total = fetch_laps(season, rnd)
            lap_frames.append(laps_df)
            qc_rows.append({
                "season": season, "round": rnd, "rows": len(laps_df), "api_total": api_total,
                "duplicates": int(laps_df.duplicated(KEYS).sum()),
                "laps": laps_df["lap"].max(), "drivers": laps_df["driver_id"].nunique(),
            })
        pit_frames.append(fetch_pit_stops(season, rnd))
    except RuntimeError as err:
        print("STOPPED:", err)
        incomplete = True
        break
    print(f"{season} R{rnd:>2} {race.race_name:<30} done  ({(time.time() - t0) / 60:.1f} min elapsed)")

# ---------- save and check ----------
laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)
pits_all["duration_seconds"] = pits_all["duration_raw"].apply(to_seconds)

laps_all.to_csv(PROCESSED_DIR / "laps_all_2023_2024.csv", index=False)
pits_all.to_csv(PROCESSED_DIR / "pit_stops_all_2023_2024.csv", index=False)

qc = pd.DataFrame(qc_rows)
qc.to_csv(PROCESSED_DIR / "fetch_qc_2023_2024.csv", index=False)

print("\nraces with laps:", laps_all.groupby(["season", "round"]).ngroups,
      "| lap rows:", len(laps_all),
      "| duplicate lap keys:", int(laps_all.duplicated(KEYS).sum()))
if len(qc):
    print("races where rows != API total:\n", qc[qc["rows"] != qc["api_total"]])
print("\npit stops:", len(pits_all),
      "| duration missing in the API:", int(pits_all["duration_raw"].isna().sum()),
      "| duration in m:ss format:", int(pits_all["duration_raw"].astype(str).str.contains(":").sum()),
      "| still unparsed:", int(pits_all["duration_seconds"].isna().sum()))
print("\n2023 R3 raw durations:",
      pits_all[(pits_all["season"] == 2023) & (pits_all["round"] == 3)]["duration_raw"].head(20).tolist())
if incomplete:
    print("\nIncomplete (rate-limited). Wait ~15 minutes, then run this cell again: it resumes from the cache.")

Scheduled races: 46
2023 R 1 Bahrain Grand Prix             done  (0.0 min elapsed)
2023 R 2 Saudi Arabian Grand Prix       done  (0.0 min elapsed)
2023 R 3 Australian Grand Prix          done  (0.1 min elapsed)
2023 R 4 Azerbaijan Grand Prix          done  (0.1 min elapsed)
2023 R 5 Miami Grand Prix               done  (0.2 min elapsed)
2023 R 6 Monaco Grand Prix              done  (1.0 min elapsed)
2023 R 7 Spanish Grand Prix             done  (1.8 min elapsed)
2023 R 8 Canadian Grand Prix            done  (2.7 min elapsed)
2023 R 9 Austrian Grand Prix            done  (3.5 min elapsed)
2023 R10 British Grand Prix             done  (4.0 min elapsed)
2023 R11 Hungarian Grand Prix           done  (4.7 min elapsed)
2023 R12 Belgian Grand Prix             done  (5.2 min elapsed)
2023 R13 Dutch Grand Prix               done  (6.1 min elapsed)
2023 R14 Italian Grand Prix             done  (6.6 min elapsed)
2023 R15 Singapore Grand Prix           done  (7.3 min elapsed)
2023 R16 Japanese Gr

ValueError: could not convert string to float: ''

In [3]:
# CELL 3 — INSPECT RAW PIT-STOP DURATIONS

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"D:\projects\f1-project")
PROCESSED = ROOT / "data" / "processed"

# Reconstruct pit-stop data from the cached API responses
# First, load the CSV if it was successfully saved previously.
pit_file = PROCESSED / "pit_stops_all_2023_2024.csv"

if pit_file.exists():
    pits_all = pd.read_csv(pit_file)
    print("Loaded existing pit-stop CSV.")
else:
    print("Pit-stop CSV was not saved because the previous cell crashed.")
    print("We will reconstruct it from cached JSON in the next step.")

# Inspect malformed values if the CSV exists
if pit_file.exists() and "duration_raw" in pits_all.columns:

    durations = pits_all["duration_raw"].astype("string")

    invalid = (
        durations.isna()
        | durations.str.strip().eq("")
    )

    print("\nTotal pit-stop records:", len(pits_all))
    print("Missing or empty duration values:", invalid.sum())

    print("\nProblematic examples:")
    display(pits_all.loc[invalid].head(20))

else:
    print("\nNo existing CSV available for inspection.")

Pit-stop CSV was not saved because the previous cell crashed.
We will reconstruct it from cached JSON in the next step.

No existing CSV available for inspection.


In [4]:
import numpy as np
import pandas as pd

KEYS = ["season", "round", "lap", "driver_id"]


def to_seconds_safe(text):
    """'1:39.019' -> 99.019; '23.4' -> 23.4; None, '' or unreadable text -> NaN."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    text = str(text).strip()
    if text == "":
        return np.nan
    try:
        total = 0.0
        for part in text.split(":"):
            total = total * 60 + float(part)
        return total
    except ValueError:
        return np.nan


laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)
pits_all["duration_seconds"] = pits_all["duration_raw"].apply(to_seconds_safe)

laps_all.to_csv(PROCESSED_DIR / "laps_all_2023_2024.csv", index=False)
pits_all.to_csv(PROCESSED_DIR / "pit_stops_all_2023_2024.csv", index=False)
qc = pd.DataFrame(qc_rows)
qc.to_csv(PROCESSED_DIR / "fetch_qc_2023_2024.csv", index=False)

# ---------- laps ----------
print("LAPS")
print("races:", laps_all.groupby(["season", "round"]).ngroups,
      "| rows:", len(laps_all),
      "| duplicate keys:", int(laps_all.duplicated(KEYS).sum()))
print("new races where rows != API total:", int((qc["rows"] != qc["api_total"]).sum()), "of", len(qc))
print("new races with duplicate keys:", int((qc["duplicates"] > 0).sum()))
per_race = laps_all.groupby(["season", "round"]).agg(rows=("lap", "size"), laps=("lap", "max"), drivers=("driver_id", "nunique"))
print(per_race.describe().loc[["min", "max"]])

# ---------- pit stops ----------
raw = pits_all["duration_raw"]
print("\nPIT STOPS")
print("stops:", len(pits_all),
      "| duplicate (race, driver, stop):", int(pits_all.duplicated(["season", "round", "driver_id", "stop"]).sum()))
print("duration is None in API:", int(raw.isna().sum()),
      "| empty string:", int((raw.astype(str).str.strip() == "").sum()),
      "| m:ss format:", int(raw.astype(str).str.contains(":").sum()),
      "| unparsed after cleaning:", int(pits_all["duration_seconds"].isna().sum()))

bad = pits_all[pits_all["duration_seconds"].isna()]
print("\nraw values that stayed unparsed:", bad["duration_raw"].astype(str).value_counts().head(5).to_dict())
print("unparsed by race (top 6):")
print(bad.groupby(["season", "round"]).size().sort_values(ascending=False).head(6))

# ---------- does the new pipeline agree with the old one for the first 10 races? ----------
old_pits = pd.read_csv(PROCESSED_DIR / "pit_stops_2023_2024_first5.csv")
first10 = pits_all[[(s, r) in already_have for s, r in zip(pits_all["season"], pits_all["round"])]]
print("\nfirst 10 races | stops: old", len(old_pits), "new", len(first10),
      "| missing durations: old", int(old_pits["duration"].isna().sum()),
      "new", int(first10["duration_seconds"].isna().sum()))
print("2023 R3 raw durations:", pits_all[(pits_all["season"] == 2023) & (pits_all["round"] == 3)]["duration_raw"].head(12).tolist())

LAPS
races: 46 | rows: 50960 | duplicate keys: 0
new races where rows != API total: 0 of 36
new races with duplicate keys: 0
       rows  laps  drivers
min   815.0  44.0     16.0
max  1513.0  78.0     20.0

PIT STOPS
stops: 1737 | duplicate (race, driver, stop): 0
duration is None in API: 0 | empty string: 1 | m:ss format: 159 | unparsed after cleaning: 1

raw values that stayed unparsed: {'': 1}
unparsed by race (top 6):
season  round
2024    12       1
dtype: int64

first 10 races | stops: old 374 new 374 | missing durations: old 70 new 0
2023 R3 raw durations: ['18.056', '18.951', '18.382', '17.657', '21.659', '19.158', '17.814', '19.146', '18.319', '20.789', '15:25.181', '15:25.112']


In [5]:
# CELL 4 — RECOVER DATA AND SAVE SAFELY

import pandas as pd
import numpy as np
import json

# 1. Verify that the original fetch results still exist
required_variables = ["lap_frames", "pit_frames", "qc"]

for name in required_variables:
    if name not in globals():
        raise NameError(
            f"{name} is missing. Do not rerun the API fetch yet."
        )

print("Lap frames:", len(lap_frames))
print("Pit-stop frames:", len(pit_frames))
print("QC records:", len(qc))

# 2. Reconstruct combined datasets
laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)

# 3. Robust duration conversion
def safe_to_seconds(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    try:
        parts = text.split(":")
        total = 0.0

        for part in parts:
            total = total * 60 + float(part)

        return total

    except (ValueError, TypeError):
        return np.nan

# Preserve original duration text
pits_all["duration_seconds"] = (
    pits_all["duration_raw"].apply(safe_to_seconds)
)

# 4. Save all datasets
laps_all.to_csv(
    PROCESSED_DIR / "laps_all_2023_2024.csv",
    index=False
)

pits_all.to_csv(
    PROCESSED_DIR / "pit_stops_all_2023_2024.csv",
    index=False
)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(
    PROCESSED_DIR / "fetch_qc_2023_2024.csv",
    index=False
)

# 5. Report results
print("\nLAP DATA")
print("-" * 45)
print("Rows:", len(laps_all))
print("Unique races:", laps_all.groupby(["season", "round"]).ngroups)
print("Duplicate lap keys:",
      laps_all.duplicated(KEYS).sum())

print("\nPIT-STOP DATA")
print("-" * 45)
print("Records:", len(pits_all))
print("Unique races:", pits_all.groupby(["season", "round"]).ngroups)
print("Missing raw durations:", pits_all["duration_raw"].isna().sum())
print("Empty raw durations:",
      pits_all["duration_raw"].astype("string").str.strip().eq("").sum())
print("Unparsed durations:", pits_all["duration_seconds"].isna().sum())

print("\nSAVED FILES")
print("-" * 45)
print("laps_all_2023_2024.csv")
print("pit_stops_all_2023_2024.csv")
print("fetch_qc_2023_2024.csv")

Lap frames: 37
Pit-stop frames: 46
QC records: 36

LAP DATA
---------------------------------------------
Rows: 50960
Unique races: 46
Duplicate lap keys: 0

PIT-STOP DATA
---------------------------------------------
Records: 1737
Unique races: 46
Missing raw durations: 0
Empty raw durations: 1
Unparsed durations: 1

SAVED FILES
---------------------------------------------
laps_all_2023_2024.csv
pit_stops_all_2023_2024.csv
fetch_qc_2023_2024.csv


In [6]:
import numpy as np
import pandas as pd

# 1. Did every newly fetched race return as many rows as the API reported?
mismatch = qc_df[qc_df["rows"] != qc_df["api_total"]]
print("Newly fetched races:", len(qc_df), "| row count != API total:", len(mismatch))
if len(mismatch):
    print(mismatch)

# 2. Per-race sanity: laps, drivers, rows
per_race = (laps_all.groupby(["season", "round"])
            .agg(rows=("lap", "size"), max_lap=("lap", "max"), drivers=("driver_id", "nunique"))
            .reset_index())
print("\nPer-race summary (min / median / max):")
print(per_race[["rows", "max_lap", "drivers"]].describe().loc[["min", "50%", "max"]].round(0))
print("\nRaces with fewer than 18 drivers or fewer than 40 laps:")
print(per_race[(per_race["drivers"] < 18) | (per_race["max_lap"] < 40)])

# 3. Pit durations: what do normal and very long stops look like?
dur = pits_all["duration_seconds"].dropna()
print("\nPit duration (s) percentiles:")
print(dur.quantile([0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))
print("stops longer than 60 s:", int((dur > 60).sum()))
long_by_race = (pits_all[pits_all["duration_seconds"] > 60]
                .groupby(["season", "round"]).size().sort_values(ascending=False))
print("long stops by race:\n", long_by_race.head(8))

# 4. The one stop with no duration
print("\nStops with no parsed duration:")
print(pits_all[pits_all["duration_seconds"].isna()])

Newly fetched races: 36 | row count != API total: 0

Per-race summary (min / median / max):
       rows  max_lap  drivers
min   815.0     44.0     16.0
50%  1071.0     58.0     20.0
max  1513.0     78.0     20.0

Races with fewer than 18 drivers or fewer than 40 laps:
    season  round  rows  max_lap  drivers
19    2023     20  1106       71       17
29    2024      8  1233       78       16

Pit duration (s) percentiles:
0.01      17.4
0.25      22.1
0.50      23.9
0.75      28.8
0.95    1411.2
0.99    2394.5
Name: duration_seconds, dtype: float64
stops longer than 60 s: 159
long stops by race:
 season  round
2023    3        49
        19       18
2024    4        18
2023    13       17
        20       17
2024    21       17
        8        16
2023    16        1
dtype: int64

Stops with no parsed duration:
      season  round driver_id  lap  stop      time duration_raw  \
1327    2024     12   tsunoda   27     1  15:45:52                

      duration_seconds  
1327             

In [7]:
# CELL 5 — FINAL EXPANDED DATASET AUDIT

import pandas as pd
import numpy as np

PROCESSED_DIR = ROOT / "data" / "processed"

laps_all = pd.read_csv(
    PROCESSED_DIR / "laps_all_2023_2024.csv"
)

pits_all = pd.read_csv(
    PROCESSED_DIR / "pit_stops_all_2023_2024.csv"
)

qc = pd.read_csv(
    PROCESSED_DIR / "fetch_qc_2023_2024.csv"
)

# Expected race counts
expected_races = {
    2023: 22,
    2024: 24
}

print("RACE COVERAGE")
print("-" * 50)

for season, expected in expected_races.items():
    actual = laps_all[laps_all["season"] == season]["round"].nunique()

    print(
        f"{season}: {actual}/{expected} races",
        "| Status:",
        "OK" if actual == expected else "CHECK"
    )

# Check lap uniqueness
lap_duplicates = laps_all.duplicated(
    ["season", "round", "lap", "driver_id"]
).sum()

# Check pit-stop uniqueness
pit_duplicates = pits_all.duplicated(
    ["season", "round", "driver_id", "stop"]
).sum()

# Missing durations by season
missing_duration = pits_all[
    pits_all["duration_seconds"].isna()
].groupby("season").size()

print("\nDATA INTEGRITY")
print("-" * 50)
print("Lap rows:", len(laps_all))
print("Lap duplicates:", lap_duplicates)
print("Pit-stop rows:", len(pits_all))
print("Pit-stop duplicates:", pit_duplicates)
print("Missing parsed durations:", pits_all["duration_seconds"].isna().sum())

print("\nMISSING DURATIONS BY SEASON")
print(missing_duration)

print("\nFETCH QC")
print("-" * 50)
print("QC rows:", len(qc))
print("Mismatched race counts:", (qc["rows"] != qc["api_total"]).sum())

print("\nAUDIT COMPLETE")

RACE COVERAGE
--------------------------------------------------
2023: 22/22 races | Status: OK
2024: 24/24 races | Status: OK

DATA INTEGRITY
--------------------------------------------------
Lap rows: 50960
Lap duplicates: 0
Pit-stop rows: 1737
Pit-stop duplicates: 0
Missing parsed durations: 1

MISSING DURATIONS BY SEASON
season
2024    1
dtype: int64

FETCH QC
--------------------------------------------------
QC rows: 36
Mismatched race counts: 0

AUDIT COMPLETE
